# CLEVR-N Relational Binding

This notebook demonstrates how PRINet uses oscillatory binding to solve
relational reasoning tasks on the synthetic CLEVR-N benchmark.

CLEVR-N presents scenes with $N$ objects and pairwise spatial queries
("is object A left of object B?"). PRINet's phase synchronization
binds object representations, enabling relational inference.

**Prerequisites:** Install PRINet from source (`pip install -e ".[dev]"`)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from prinet.nn.hybrid import HybridCLEVRN, HybridPRINetV2CLEVRN
from prinet.utils.y4q1_tools import train_clevr_n_single_seed
from prinet import kuramoto_order_parameter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Model Architecture

PRINet offers two CLEVR-N architectures:

- **HybridCLEVRN**: Uses multi-band oscillators (delta/theta/gamma)
  with learned-oscillatory-binding-and-mixture (LOBM)
- **HybridPRINetV2CLEVRN**: Adds PAC (phase-amplitude coupling)
  and discrete delta-theta-gamma dynamics

Both take scene features + query and produce a binary classification.

In [ ]:
# Create the basic model
model_basic = HybridCLEVRN(
    scene_dim=16,
    query_dim=44,
    n_delta=4,
    n_theta=8,
    n_gamma=32,
    hidden_dim=64,
    lobm_steps=5,
)

n_params = sum(p.numel() for p in model_basic.parameters())
print(f"HybridCLEVRN: {n_params:,} parameters")

# Create the V2 model with PAC
model_v2 = HybridPRINetV2CLEVRN(
    scene_dim=16,
    query_dim=60,
    n_delta=4,
    n_theta=8,
    n_gamma=32,
    d_model=64,
    n_discrete_steps=5,
    coupling_strength=2.0,
    pac_depth=0.3,
)

n_params_v2 = sum(p.numel() for p in model_v2.parameters())
print(f"HybridPRINetV2CLEVRN: {n_params_v2:,} parameters")

## 2. Training on CLEVR-N

Train both models on CLEVR-6 (6 objects per scene) for 50 epochs:

In [ ]:
# Train HybridCLEVRN
model_basic_train = HybridCLEVRN(
    scene_dim=16, query_dim=44,
    n_delta=4, n_theta=8, n_gamma=32,
    hidden_dim=64, lobm_steps=5,
).to(DEVICE)

acc_basic, loss_basic, time_basic = train_clevr_n_single_seed(
    model_basic_train,
    n_objects=6,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    seed=42,
    device=DEVICE,
)

print(f"HybridCLEVRN:     acc={acc_basic:.1%}  loss={loss_basic:.4f}  time={time_basic:.1f}s")

In [ ]:
# Train HybridPRINetV2CLEVRN
model_v2_train = HybridPRINetV2CLEVRN(
    scene_dim=16, query_dim=60,
    n_delta=4, n_theta=8, n_gamma=32,
    d_model=64, n_discrete_steps=5,
    coupling_strength=2.0, pac_depth=0.3,
).to(DEVICE)

acc_v2, loss_v2, time_v2 = train_clevr_n_single_seed(
    model_v2_train,
    n_objects=6,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    seed=42,
    device=DEVICE,
)

print(f"HybridPRINetV2CLEVRN: acc={acc_v2:.1%}  loss={loss_v2:.4f}  time={time_v2:.1f}s")

## 3. Scaling with Object Count

One of PRINet's key claims is $O(N)$ binding cost. Let's measure
accuracy vs. number of objects:

In [ ]:
n_objects_list = [3, 4, 6, 8, 10]
results = {}

for n_obj in n_objects_list:
    model = HybridCLEVRN(
        scene_dim=16, query_dim=44,
        n_delta=4, n_theta=8, n_gamma=32,
        hidden_dim=64, lobm_steps=5,
    ).to(DEVICE)
    acc, loss, t = train_clevr_n_single_seed(
        model, n_objects=n_obj, n_epochs=50,
        batch_size=64, lr=1e-3, seed=42, device=DEVICE,
    )
    results[n_obj] = (acc, loss, t)
    print(f"N={n_obj:2d}: acc={acc:.1%}  loss={loss:.4f}  time={t:.1f}s")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n_objects_list, [results[n][0] for n in n_objects_list], "o-")
axes[0].set_xlabel("Number of objects")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("CLEVR-N Accuracy vs. Object Count")
axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_objects_list, [results[n][2] for n in n_objects_list], "s-", color="orange")
axes[1].set_xlabel("Number of objects")
axes[1].set_ylabel("Training time (s)")
axes[1].set_title("Wall-Clock Time vs. Object Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Oscillator Phase Dynamics During Binding

Visualize how oscillator phases evolve during forward pass to
understand the binding mechanism:

In [ ]:
from prinet import KuramotoOscillator, OscillatorState

# Simulate binding: N oscillators, 3 "objects" of different frequencies
N = 24
state = OscillatorState.create_random(N, freq_range=(1.0, 10.0), seed=42)

# Assign 3 frequency clusters to mimic object binding
state.frequency[:8] = 2.0    # Object A
state.frequency[8:16] = 5.0  # Object B
state.frequency[16:] = 8.0   # Object C

osc = KuramotoOscillator(
    n_oscillators=N,
    coupling_strength=3.0,
    coupling_mode="mean_field",
)

# Record phase evolution
phases = [state.phase.clone().cpu().numpy()]
for _ in range(200):
    state = osc.step(state, dt=0.01)
    phases.append(state.phase.clone().cpu().numpy())

phases = np.array(phases)  # (201, 24)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = ['#e74c3c'] * 8 + ['#2ecc71'] * 8 + ['#3498db'] * 8
for i in range(N):
    axes[0].plot(phases[:, i], color=colors[i], alpha=0.4, linewidth=0.8)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Phase (rad)")
axes[0].set_title("Phase trajectories (color=object)")

# Order parameter per cluster
for label, sl, c in [("Obj A", slice(0, 8), "#e74c3c"),
                      ("Obj B", slice(8, 16), "#2ecc71"),
                      ("Obj C", slice(16, 24), "#3498db")]:
    r_vals = []
    for t in range(len(phases)):
        ph = torch.tensor(phases[t, sl])
        r_vals.append(kuramoto_order_parameter(ph).item())
    axes[1].plot(r_vals, label=label, color=c)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("r")
axes[1].set_title("Per-object synchronization")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Final phase polar plot
ax_polar = plt.subplot(1, 3, 3, projection='polar')
for i in range(N):
    ax_polar.scatter(phases[-1, i], 1.0, c=colors[i], s=40, zorder=3)
ax_polar.set_title("Final phases (polar)", pad=15)

plt.tight_layout()
plt.show()

## Summary

| Experiment | Key Insight |
|-----------|------------|
| Model comparison | HybridPRINetV2CLEVRN adds PAC for cross-frequency binding |
| Object scaling | PRINet maintains accuracy as $N$ grows |
| Phase dynamics | Oscillators belonging to the same object synchronize |

The binding mechanism is fundamentally different from attention:
objects are represented as *phase-coherent clusters* rather than
weighted sums of key-value pairs.